# 05 - Aggregate Results and Statistics

This notebook converts raw evaluation outputs into the final comparison tables.

What this notebook does:

- summarizes default and tuned final returns
- computes IQM, confidence intervals, and seed variability
- measures sample efficiency at intermediate budgets
- builds stability tables using fall rate and episode length
- computes probability-of-improvement matrices
- produces the overall algorithm ranking

These tables are the main numerical evidence used in the report.

## Load data

The aggregation notebook starts from the saved final-evaluation files,
robustness files, and training-curve CSVs. These are the raw materials for the
tables used in the report.

In [1]:
from pathlib import Path
import os, sys, time, json

def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path(r"D:/MuJoCo_RL_Project")]:
        if (candidate / "project_utils.py").exists():
            return candidate.resolve()
    raise RuntimeError("project_utils.py was not found")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

import project_utils as pu
pu.set_plot_style()
pu.ensure_dirs()

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")


[project_utils] Loaded - Project root: D:\MuJoCo_RL_Project
  Python 3.10.20 | SB3 2.9.0 | Gymnasium 1.3.0
  Torch 2.12.1+cu126 | CUDA: True | GPU: NVIDIA RTX 5000 Ada Generation
Project root: D:\MuJoCo_RL_Project
Python: C:\Users\digilians01\.conda\envs\RL_PROJECT\python.exe


In [2]:
final_eval = pd.read_csv(pu.RESULTS_PROCESSED / "final_eval_all.csv")
robust_path = pu.RESULTS_PROCESSED / "robustness_eval_all.csv"
robust = pd.read_csv(robust_path) if robust_path.exists() else pd.DataFrame()

train_frames = []
for path in pu.RESULTS_RAW.glob("*__train_eval.csv"):
    try:
        train_frames.append(pd.read_csv(path))
    except Exception:
        pass
train_eval = pd.concat(train_frames, ignore_index=True) if train_frames else pd.DataFrame()
print(f"Final eval rows: {len(final_eval)}")
print(f"Training eval rows: {len(train_eval)}")
print(f"Robustness rows: {len(robust)}")


Final eval rows: 96
Training eval rows: 4334
Robustness rows: 3970


## Final return tables

This section computes the main performance table for default and tuned runs. For
each algorithm and environment we report mean return, IQM return, confidence
intervals, spread across seeds, and best/worst observed seed.

In [3]:
def summarize_returns(track):
    rows = []
    data = final_eval[final_eval["track"] == track]
    for (env_id, algo), group in data.groupby(["env_id", "algorithm"]):
        vals = group["mean_return"].to_numpy(float)
        mean, lo, hi = pu.bootstrap_ci(vals, np.mean, n_bootstrap=2000)
        iqm, iqm_lo, iqm_hi = pu.bootstrap_ci(vals, pu.compute_iqm, n_bootstrap=2000)
        rows.append({
            "track": track, "env_id": env_id, "algorithm": algo,
            "n_seeds": len(vals), "mean_return": mean,
            "mean_ci95_low": lo, "mean_ci95_high": hi,
            "median_return": float(np.median(vals)),
            "iqm_return": iqm, "iqm_ci95_low": iqm_lo,
            "iqm_ci95_high": iqm_hi, "std_return": float(np.std(vals)),
            "min_return": float(np.min(vals)), "max_return": float(np.max(vals)),
            "cv": float(np.std(vals) / abs(np.mean(vals))) if abs(np.mean(vals)) > 1e-9 else np.nan,
        })
    return pd.DataFrame(rows)

default_path = pu.RESULTS_FINAL / "table_default_1m_final_returns.csv"
tuned_path = pu.RESULTS_FINAL / "table_tuned_1m_final_returns.csv"
if default_path.exists():
    default_stats = pd.read_csv(default_path)
else:
    default_stats = summarize_returns("default_1m")
    default_stats.to_csv(default_path, index=False)
if tuned_path.exists():
    tuned_stats = pd.read_csv(tuned_path)
else:
    tuned_stats = summarize_returns("tuned_1m")
    tuned_stats.to_csv(tuned_path, index=False)
display(tuned_stats.round(3))


,track,env_id,algorithm,n_seeds,mean_return,mean_ci95_low,mean_ci95_high,median_return,median_ci95_low,median_ci95_high,iqm_return,iqm_ci95_low,iqm_ci95_high,std_return,min_return,max_return,cv
0,tuned_1m,HalfCheetah-v5,PPO,3,1367.046,1020.286,1625.379,1455.474,1020.286,1625.379,1367.046,1020.286,1625.379,254.819,1020.286,1625.379,0.186
1,tuned_1m,HalfCheetah-v5,SAC,3,11021.715,10778.030,11225.269,11061.848,10778.030,11225.269,11021.715,10778.030,11225.269,184.776,10778.030,11225.269,0.017
2,tuned_1m,HalfCheetah-v5,TD3,3,10391.186,9757.287,10811.009,10605.261,9757.287,10811.009,10391.186,9757.287,10811.009,456.036,9757.287,10811.009,0.044
3,tuned_1m,HalfCheetah-v5,DDPG,3,9011.620,8430.835,9686.275,8917.749,8430.835,9686.275,9011.620,8430.835,9686.275,516.812,8430.835,9686.275,0.057
4,tuned_1m,HalfCheetah-v5,TQC,3,12175.689,12150.146,12220.185,12156.736,12150.146,12220.185,12175.689,12150.146,12220.185,31.578,12150.146,12220.185,0.003
5,tuned_1m,Hopper-v5,PPO,3,2279.567,1546.969,2691.944,2599.789,1546.969,2691.944,2279.567,1546.969,2691.944,519.389,1546.969,2691.944,0.228
6,tuned_1m,Hopper-v5,SAC,3,3385.413,3245.495,3488.350,3422.395,3245.495,3488.350,3385.413,3245.495,3488.350,102.536,3245.495,3488.350,0.030
7,tuned_1m,Hopper-v5,TD3,3,3626.094,3605.766,3648.540,3623.977,3605.766,3648.540,3626.094,3605.766,3648.540,17.526,3605.766,3648.540,0.005
8,tuned_1m,Hopper-v5,DDPG,3,2610.941,2576.052,2659.699,2597.072,2576.052,2659.699,2610.941,2576.052,2659.699,35.529,2576.052,2659.699,0.014
9,tuned_1m,Hopper-v5,TQC,3,2599.745,1958.010,3544.418,2296.806,1958.010,3544.418,2599.745,1958.010,3544.418,682.154,1958.010,3544.418,0.262


## Sample efficiency

Sample efficiency asks how much return the agent gets at intermediate training
budgets. We measure returns near 100k, 250k, 500k, and 1M timesteps using the
periodic training-evaluation logs.

In [4]:
checkpoints = [100_000, 250_000, 500_000, 1_000_000]
rows = []
for (env_id, algo), group in train_eval.groupby(["env_id", "algorithm"]):
    row = {"env_id": env_id, "algorithm": algo}
    for step in checkpoints:
        near = group[np.abs(group["timestep"] - step) <= step * 0.10]
        row[f"return_at_{step//1000}k"] = near["eval_mean_return"].mean() if not near.empty else np.nan
    rows.append(row)
sample_path = pu.RESULTS_FINAL / "table_sample_efficiency.csv"
if sample_path.exists():
    sample_eff = pd.read_csv(sample_path)
else:
    sample_eff = pd.DataFrame(rows)
    sample_eff.to_csv(sample_path, index=False)
display(sample_eff.round(2))


,env_id,algorithm,return_at_100k,return_at_250k,return_at_500k,return_at_1000k
0,HalfCheetah-v5,PPO,416.09,1198.97,1314.50,1459.01
1,HalfCheetah-v5,SAC,4663.69,6958.90,8524.26,9691.24
2,HalfCheetah-v5,TD3,4890.96,6826.70,9233.63,10318.85
3,HalfCheetah-v5,DDPG,4008.29,5381.99,7139.33,8127.52
4,HalfCheetah-v5,TQC,4629.67,7428.74,9246.41,11113.24
5,Hopper-v5,PPO,1200.19,2036.36,2001.29,2254.17
6,Hopper-v5,SAC,579.62,1884.18,2286.45,2651.67
7,Hopper-v5,TD3,1814.48,2793.30,3536.94,3433.64
8,Hopper-v5,DDPG,388.61,1281.05,1402.84,1653.87
9,Hopper-v5,TQC,1060.98,2864.17,3012.62,2662.20


## Stability

Stability is measured with seed variability, fall rate, and episode length.
Algorithms with high average return but high variance or frequent falls are less
reliable than algorithms that perform consistently across seeds.

In [5]:
stability_path = pu.RESULTS_FINAL / "table_stability.csv"
if stability_path.exists():
    stability = pd.read_csv(stability_path)
else:
    rows = []
    for (track, env_id, algo), group in final_eval.groupby(["track", "env_id", "algorithm"]):
        vals = group["mean_return"].to_numpy(float)
        rows.append({
            "track": track, "env_id": env_id, "algorithm": algo,
            "n_seeds": len(vals), "mean_return": np.mean(vals),
            "std_return": np.std(vals),
            "cv": np.std(vals) / abs(np.mean(vals)) if abs(np.mean(vals)) > 1e-9 else np.nan,
            "worst_seed_return": np.min(vals), "best_seed_return": np.max(vals),
            "mean_fall_rate": group["fall_rate"].mean(),
            "mean_ep_length": group["mean_ep_length"].mean(),
        })
    stability = pd.DataFrame(rows)
    stability.to_csv(stability_path, index=False)
display(stability.round(3))


,track,env_id,algorithm,n_seeds,mean_return,std_return,cv,iqr,worst_seed_return,best_seed_return,collapse_count,collapse_rate,survival_collapse_count,mean_fall_rate,mean_ep_length
0,default_1m,HalfCheetah-v5,PPO,3,1620.147,71.272,0.044,79.588,1519.861,1679.037,0,0.0,0,0.000,1000.000
1,default_1m,HalfCheetah-v5,SAC,5,8852.285,2134.574,0.241,3548.597,6737.371,12166.373,0,0.0,0,0.000,1000.000
2,default_1m,HalfCheetah-v5,TD3,3,10479.534,511.279,0.049,556.510,9757.287,10870.307,0,0.0,0,0.000,1000.000
3,default_1m,HalfCheetah-v5,DDPG,3,7424.107,941.303,0.127,1006.095,6093.024,8105.215,0,0.0,0,0.000,1000.000
4,default_1m,HalfCheetah-v5,TQC,3,10399.716,1107.248,0.106,1202.160,9559.869,11964.188,0,0.0,0,0.000,1000.000
5,default_1m,Hopper-v5,PPO,3,1670.549,485.690,0.291,582.173,1017.861,2182.207,0,0.0,0,0.950,452.250
6,default_1m,Hopper-v5,SAC,5,2144.544,817.808,0.381,1287.609,1216.330,3404.839,0,0.0,0,0.610,636.600
7,default_1m,Hopper-v5,TD3,3,3626.094,17.526,0.005,21.387,3605.766,3648.540,0,0.0,0,0.033,994.917
8,default_1m,Hopper-v5,DDPG,3,2212.721,875.462,0.396,1064.663,1074.700,3204.026,0,0.0,0,0.550,644.583
9,default_1m,Hopper-v5,TQC,3,3196.234,75.693,0.024,82.253,3138.670,3303.176,0,0.0,0,0.283,903.967


## Probability of improvement

Probability of improvement compares algorithms pairwise. A value above 0.5 means
the row algorithm more often outperforms the column algorithm on the sampled
evaluation returns.

In [6]:
poi_rows = []
for env_id in pu.ENV_IDS:
    matrix_path = pu.RESULTS_FINAL / f"poi_matrix_{env_id.replace('-', '_')}.csv"
    if matrix_path.exists():
        matrix = pd.read_csv(matrix_path, index_col=0)
    else:
        env_data = final_eval[(final_eval["track"] == "tuned_1m") & (final_eval["env_id"] == env_id)]
        matrix = pd.DataFrame(index=pu.ALGORITHMS, columns=pu.ALGORITHMS, dtype=float)
        for a in pu.ALGORITHMS:
            for b in pu.ALGORITHMS:
                scores_a = env_data[env_data["algorithm"] == a]["mean_return"]
                scores_b = env_data[env_data["algorithm"] == b]["mean_return"]
                matrix.loc[a, b] = pu.probability_of_improvement(scores_a, scores_b)
        matrix.to_csv(matrix_path)
    print(env_id)
    display(matrix.round(3))

poi_table = pu.RESULTS_FINAL / "table_probability_of_improvement.csv"
if not poi_table.exists():
    for env_id in pu.ENV_IDS:
        matrix = pd.read_csv(pu.RESULTS_FINAL / f"poi_matrix_{env_id.replace('-', '_')}.csv", index_col=0)
        for a in matrix.index:
            for b in matrix.columns:
                poi_rows.append({"env_id": env_id, "algorithm_a": a, "algorithm_b": b,
                                 "probability_a_beats_b": matrix.loc[a, b]})
    pd.DataFrame(poi_rows).to_csv(poi_table, index=False)


HalfCheetah-v5


,PPO,SAC,TD3,DDPG,TQC
PPO,0.5,0.000,0.000,0.000,0.000
SAC,1.0,0.500,0.144,0.955,0.027
TD3,1.0,0.856,0.500,1.000,0.053
DDPG,1.0,0.045,0.000,0.500,0.000
TQC,1.0,0.973,0.947,1.000,0.500


Hopper-v5


,PPO,SAC,TD3,DDPG,TQC
PPO,0.500,0.058,0.0,0.116,0.003
SAC,0.942,0.500,0.0,0.686,0.238
TD3,1.000,1.000,0.5,1.000,1.000
DDPG,0.884,0.314,0.0,0.500,0.080
TQC,0.997,0.762,0.0,0.920,0.500


Walker2d-v5


,PPO,SAC,TD3,DDPG,TQC
PPO,0.500,0.000,0.000,0.495,0.000
SAC,1.000,0.500,0.099,1.000,0.003
TD3,1.000,0.901,0.500,1.000,0.070
DDPG,0.505,0.000,0.000,0.500,0.000
TQC,1.000,0.997,0.930,1.000,0.500


## Overall ranking

This section turns the per-environment results into an overall ranking. The
ranking is based on within-environment performance so one high-scale environment
does not dominate the comparison.

In [7]:
ranking_path = pu.RESULTS_FINAL / "table_overall_ranking.csv"
if ranking_path.exists():
    overall = pd.read_csv(ranking_path)
else:
    rank_df = tuned_stats.copy()
    rank_df["rank_iqm"] = rank_df.groupby("env_id")["iqm_return"].rank(ascending=False)
    rank_df["rank_mean"] = rank_df.groupby("env_id")["mean_return"].rank(ascending=False)
    overall = (rank_df.groupby("algorithm")
               .agg(avg_rank_iqm=("rank_iqm", "mean"),
                    avg_rank_mean=("rank_mean", "mean"))
               .sort_values("avg_rank_iqm")
               .reset_index())
    overall.to_csv(ranking_path, index=False)
display(overall)


,algorithm,avg_rank_iqm,avg_rank_mean
0,TQC,1.333333,1.333333
1,TD3,1.666667,1.666667
2,SAC,3.000000,3.000000
3,DDPG,4.000000,4.333333
4,PPO,5.000000,4.666667


## Saved final tables

This final check lists the tables written for the report and confirms that they
exist on disk.

In [8]:
for name in [
    "table_default_1m_final_returns.csv",
    "table_tuned_1m_final_returns.csv",
    "table_sample_efficiency.csv",
    "table_stability.csv",
    "table_probability_of_improvement.csv",
    "table_overall_ranking.csv",
]:
    path = pu.RESULTS_FINAL / name
    print(path.name, path.exists(), path.stat().st_size if path.exists() else 0)


table_default_1m_final_returns.csv True 4057
table_tuned_1m_final_returns.csv True 3996
table_sample_efficiency.csv True 1402
table_stability.csv True 2699
table_probability_of_improvement.csv True 1863
table_overall_ranking.csv True 192
